In [ ]:
#%% [markdown]
# # Credit Card Fraud Detection with Explainable AI (XGBoost & Modified Rain Optimization)

# This notebook demonstrates a complete workflow for building a credit card fraud detection system using XGBoost. It includes:
# 1.  **Data Loading and Exploration (EDA)**: Understanding the dataset characteristics.
# 2.  **Data Preprocessing**: Handling features, scaling, and addressing class imbalance using SMOTE.
# 3.  **Feature Selection**: Identifying important features using RandomForest.
# 4.  **Hyperparameter Tuning**: Optimizing XGBoost using a custom Modified Rain Optimization algorithm.
# 5.  **Model Training**: Training the final XGBoost model with optimized parameters.
# 6.  **Model Evaluation**: Assessing performance using various metrics (Classification Report, Confusion Matrix, ROC AUC, Precision-Recall AUC) and finding an optimal threshold.
# 7.  **Explainable AI (XAI)**: Using SHAP and LIME to understand model predictions.
# 8.  **System Implementation**: Defining a function to use the model for detecting fraud in new transactions and providing explanations.
# 9.  **Saving Components**: Persisting the trained model and necessary preprocessing objects.

#%% [markdown]
# ## 1. Imports and Setup

#%%

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb # import the xgboost library and alias it as xgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTE
import shap
import lime
import lime.lime_tabular
from scipy import stats
import warnings
import random
from datetime import datetime, timedelta
import time
import joblib # For saving the model and components

warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Configure plotting style
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
#%% [markdown]
# ## 2. Modified Rain Optimization Algorithm (RO) for Hyperparameter Tuning
# This custom class implements a nature-inspired optimization algorithm to find the best hyperparameters for the XGBoost model.

#%%
class ModifiedRainOptimization:
    def __init__(self, param_grid, scoring='roc_auc', n_iterations=20, population_size=10):
        self.param_grid = param_grid
        self.n_iterations = n_iterations
        self.population_size = population_size
        self.scoring = scoring
        self.best_params_ = None
        self.best_score_ = -np.inf
        self.history = []

    def _generate_random_solution(self):
        """Generate a random solution from the parameter grid"""
        solution = {}
        for param, values in self.param_grid.items():
            solution[param] = random.choice(values)
        return solution

    def _evaluate_solution(self, solution, X, y):
        """Evaluate a solution using cross-validation"""
        model = XGBClassifier(**solution, random_state=42, use_label_encoder=False, eval_metric='logloss') # Added common XGBoost args
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        # Use try-except block for robustness during cross-validation
        try:
            scores = cross_val_score(model, X, y, cv=skf, scoring=self.scoring, n_jobs=-1) # Use multiple cores
            return np.mean(scores)
        except Exception as e:
            print(f"Warning: Cross-validation failed for params {solution}. Error: {e}. Returning -inf.")
            return -np.inf # Return a very low score if CV fails

    def _update_solution(self, solution, best_solution, iteration):
        """Update a solution based on the modified rain optimization algorithm"""
        updated_solution = solution.copy()

        # Define dynamic parameters for the algorithm
        evaporation_rate = 0.85 - 0.8 * (iteration / self.n_iterations)
        precipitation_factor = 0.2 + 0.6 * (iteration / self.n_iterations)

        for param, values in self.param_grid.items():
            # Evaporation phase (exploitation) - Move towards the best known solution
            if random.random() < evaporation_rate:
                 # Small chance to randomly jump even during evaporation to avoid getting stuck
                if random.random() < 0.1:
                     updated_solution[param] = random.choice(values)
                else:
                    updated_solution[param] = best_solution[param]
            # Precipitation phase (exploration) - Explore the neighborhood
            else:
                # Ensure values is a list or tuple for indexing
                if not isinstance(values, (list, tuple)):
                    print(f"Warning: Parameter {param} values are not list/tuple: {values}. Skipping update.")
                    continue
                if not values: # Skip if values list is empty
                     print(f"Warning: Parameter {param} values list is empty. Skipping update.")
                     continue

                try:
                    current_value = solution[param]
                    if current_value not in values:
                         # If current value somehow isn't in the grid, pick a random one
                         print(f"Warning: Current value {current_value} for {param} not in grid. Resetting randomly.")
                         updated_solution[param] = random.choice(values)
                         continue

                    current_idx = values.index(current_value)
                    max_move = max(1, int(precipitation_factor * len(values) / 2)) # Adjust move range
                    move = random.randint(-max_move, max_move)
                    new_idx = max(0, min(len(values) - 1, current_idx + move))
                    updated_solution[param] = values[new_idx]
                except ValueError:
                     # Handle cases where the value might not be directly in the list (e.g., slight float differences)
                     print(f"Warning: ValueError finding index for {param}={current_value}. Resetting randomly.")
                     updated_solution[param] = random.choice(values)
                except IndexError:
                     print(f"Warning: IndexError during update for {param}. Resetting randomly.")
                     updated_solution[param] = random.choice(values)


        return updated_solution

    def fit(self, X, y):
        """Run the modified rain optimization algorithm"""
        print(f"Starting Modified Rain Optimization: {self.n_iterations} iterations, {self.population_size} population size.")
        # Generate initial population
        population = [self._generate_random_solution() for _ in range(self.population_size)]

        # Evaluate initial population
        print("Evaluating initial population...")
        scores = [self._evaluate_solution(solution, X, y) for solution in population]

        # Find best solution
        valid_scores = [s for s in scores if s > -np.inf]
        if not valid_scores:
             print("Error: Initial population evaluation failed for all solutions. Check parameters or data.")
             # Handle error appropriately - maybe raise exception or return default
             self.best_params_ = self._generate_random_solution() # Fallback
             self.best_score_ = -np.inf
        else:
            best_idx = np.argmax(scores)
            self.best_params_ = population[best_idx].copy()
            self.best_score_ = scores[best_idx]

        # Record history
        self.history.append({
            'iteration': 0,
            'best_score': self.best_score_,
            'best_params': self.best_params_
        })
        print(f"Iteration 0: Best Score = {self.best_score_:.4f}")

        # Iterate
        for i in range(1, self.n_iterations + 1):
            start_iter_time = time.time()
            # Update population
            new_population = []
            for j in range(self.population_size):
                # Ensure population[j] exists and is valid before updating
                if j < len(population) and isinstance(population[j], dict):
                     new_solution = self._update_solution(population[j], self.best_params_, i)
                     new_population.append(new_solution)
                else:
                     print(f"Warning: Invalid individual at index {j} in iteration {i}. Regenerating.")
                     new_population.append(self._generate_random_solution())


            # Evaluate new population
            new_scores = [self._evaluate_solution(solution, X, y) for solution in new_population]

            # Update best solution found so far
            current_best_iter_score = -np.inf
            current_best_iter_params = None
            for j in range(self.population_size):
                 if j < len(new_scores) and new_scores[j] > self.best_score_:
                     self.best_params_ = new_population[j].copy()
                     self.best_score_ = new_scores[j]
                 # Track the best score within this specific iteration as well
                 if j < len(new_scores) and new_scores[j] > current_best_iter_score:
                      current_best_iter_score = new_scores[j]
                      current_best_iter_params = new_population[j]


            # Record history (best overall score)
            self.history.append({
                'iteration': i,
                'best_score': self.best_score_,
                'best_params': self.best_params_
            })

            # Update population for the next iteration (replace worst with best if needed, or just carry over)
            # Simple carry-over strategy used here:
            population = new_population
            scores = new_scores # Keep track of current population scores (though not strictly necessary with simple carry-over)

            end_iter_time = time.time()
            print(f"Iteration {i}: Best Score = {self.best_score_:.4f}, Iter Time = {end_iter_time - start_iter_time:.2f}s")


        print("Modified Rain Optimization finished.")
        return self




In [ ]:
#%% [markdown]
# ## 3. Load the Dataset

#%%
print("Step 1: Loading the Credit Card Fraud Dataset...")
try:
    data = pd.read_csv('creditcard.csv')
    print("Dataset loaded successfully.")
    print(f"Dataset Shape: {data.shape}")
except FileNotFoundError:
    print("Error: creditcard.csv not found. Please ensure the file is in the correct directory.")
    # Exit or handle the error appropriately in a real notebook scenario
    data = None # Set data to None to prevent further errors

#%% [markdown]
# ## 4. Exploratory Data Analysis (EDA)

#%%
if data is not None:
    print("Step 2: Performing Exploratory Data Analysis...")

    # Dataset information
    print("\nDataset Information:")
    print(f"Total number of transactions: {len(data)}")
    fraud_count = len(data[data['Class'] == 1])
    normal_count = len(data[data['Class'] == 0])
    print(f"Fraud transactions: {fraud_count}")
    print(f"Normal transactions: {normal_count}")
    if len(data) > 0:
        print(f"Fraud percentage: {fraud_count / len(data) * 100:.4f}%")
    else:
        print("Fraud percentage: N/A (empty dataset)")

    # Check for missing values
    missing_values = data.isnull().sum()
    print("\nChecking for missing values:")
    if missing_values.sum() > 0:
        print(missing_values[missing_values > 0])
    else:
        print("No missing values found.")

    # Descriptive statistics for normal and fraud transactions
    print("\nDescriptive Statistics for Normal Transactions:")
    print(data[data['Class'] == 0]['Amount'].describe()) # Focus on Amount initially
    print("\nDescriptive Statistics for Fraud Transactions:")
    print(data[data['Class'] == 1]['Amount'].describe()) # Focus on Amount initially

    # It might be too verbose to print describe() for all V columns here.
    # Consider specific analysis if needed.

In [ ]:
#%% [markdown]
# ### 4.1 Visualize Class Imbalance

#%%
if data is not None:
    plt.figure(figsize=(8, 5))
    sns.countplot(x='Class', data=data, palette='viridis')
    plt.title('Class Distribution (0: Normal, 1: Fraud)')
    plt.xlabel('Class')
    plt.ylabel('Frequency')
    plt.xticks([0, 1], ['Normal', 'Fraud'])
    plt.tight_layout()
    plt.savefig('class_distribution.png')
    plt.show()

In [ ]:
#%% [markdown]
# ### 4.2 Visualize Transaction Amount Distribution

#%%
if data is not None:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # Boxplot for Amount by Class
    sns.boxplot(x='Class', y='Amount', data=data, ax=axes[0], palette='viridis', showfliers=False) # Hide outliers for better scale
    axes[0].set_title('Amount Distribution by Class (Outliers Hidden)')
    axes[0].set_xticks([0, 1], ['Normal', 'Fraud'])
    axes[0].set_ylabel('Amount')
    axes[0].set_xlabel('Class')


    # Histplot for Amount (log scale often useful for skewed data like amount)
    # Plotting distributions separately for clarity due to scale difference
    sns.histplot(data[data['Class'] == 0]['Amount'] + 1e-6, kde=False, bins=50, color='blue', alpha=0.6, label='Normal', ax=axes[1])
    sns.histplot(data[data['Class'] == 1]['Amount'] + 1e-6, kde=False, bins=50, color='red', alpha=0.6, label='Fraud', ax=axes[1])
    axes[1].set_title('Amount Distribution (Log Scale)')
    axes[1].set_yscale('log') # Use log scale for y-axis (frequency)
    # axes[1].set_xscale('log') # Optionally use log scale for x-axis (amount) if very skewed
    axes[1].set_xlabel('Amount (Log Scale on Y-axis)')
    axes[1].set_ylabel('Frequency (Log Scale)')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('amount_distribution.png')
    plt.show()

    # Additional plot for Amount comparison with log transform may be insightful
    plt.figure(figsize=(10, 6))
    sns.histplot(np.log1p(data.loc[data['Class'] == 0, 'Amount']), label='Normal', kde=True, color='blue', bins=50)
    sns.histplot(np.log1p(data.loc[data['Class'] == 1, 'Amount']), label='Fraud', kde=True, color='red', bins=50)
    plt.title('Log(Amount+1) Distribution by Class')
    plt.xlabel('Log(Amount + 1)')
    plt.legend()
    plt.tight_layout()
    plt.savefig('log_amount_distribution.png')
    plt.show()

In [ ]:
#%% [markdown]
# ### 4.3 Visualize Correlation Matrix (Selected Features)
# Showing correlation for all V1-V28 features can be overwhelming. Let's look at correlations among V1-V10 and Amount/Time.

#%%
if data is not None:
    # Select a subset of features for clarity
    cols_to_corr = ['Time', 'Amount'] + [f'V{i}' for i in range(1, 11)]
    plt.figure(figsize=(12, 10))
    correlation_matrix = data[cols_to_corr].corr()
    sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt='.2f', linewidths=.5) # Annot=False if too cluttered
    plt.title('Correlation Matrix (Time, Amount, V1-V10)')
    plt.tight_layout()
    plt.savefig('correlation_matrix_subset.png')
    plt.show()

In [ ]:
#%% [markdown]
# ## 5. Data Preprocessing

#%%
if data is not None:
    print("\nStep 3: Preprocessing the data...")

    # Separate features and target
    X = data.drop('Class', axis=1)
    y = data['Class']

    # Split the data into training and testing sets
    # Use stratify=y to maintain class proportion in splits
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"Original training set shape: {X_train.shape}")
    print(f"Testing set shape: {X_test.shape}")
    print(f"Training target distribution:\n{y_train.value_counts(normalize=True)}")
    print(f"Testing target distribution:\n{y_test.value_counts(normalize=True)}")

    # Standardize the Amount feature
    # Important: Fit scaler ONLY on training data, then transform both train and test
    print("\nStandardizing 'Amount' feature...")
    scaler = StandardScaler()
    X_train['Amount_Scaled'] = scaler.fit_transform(X_train[['Amount']])
    X_test['Amount_Scaled'] = scaler.transform(X_test[['Amount']])

    # Handle Time feature - convert to cyclical features (sin/cos of day cycle)
    # Drop original Time and Amount after creating scaled/cyclical versions
    print("Creating cyclical 'Time' features and dropping original 'Time' and 'Amount'...")
    seconds_in_day = 24 * 60 * 60
    X_train['Time_sin'] = np.sin(2 * np.pi * X_train['Time'] / seconds_in_day)
    X_train['Time_cos'] = np.cos(2 * np.pi * X_train['Time'] / seconds_in_day)
    X_test['Time_sin'] = np.sin(2 * np.pi * X_test['Time'] / seconds_in_day)
    X_test['Time_cos'] = np.cos(2 * np.pi * X_test['Time'] / seconds_in_day)

    X_train.drop(['Time', 'Amount'], axis=1, inplace=True)
    X_test.drop(['Time', 'Amount'], axis=1, inplace=True)

    print("Preprocessing steps completed.")
    print(f"Training features shape after preprocessing: {X_train.shape}")
    print(f"Testing features shape after preprocessing: {X_test.shape}")
    print("\nTraining features columns:", X_train.columns.tolist())

In [ ]:
#%% [markdown]
# ### 5.1 Apply SMOTE for Balancing the Classes
# SMOTE (Synthetic Minority Over-sampling Technique) is applied ONLY to the training data to avoid data leakage into the test set.

#%%
if 'X_train' in locals() and X_train is not None:
    print("\nApplying SMOTE to balance the training classes...")
    smote = SMOTE(random_state=42, sampling_strategy='auto') # Default is to oversample the minority class to match the majority
    
    # Ensure X_train and y_train are valid before resampling
    if X_train.isnull().sum().sum() > 0:
        print("Warning: Found NaN values in X_train before SMOTE. Consider imputation.")
        # Optionally handle NaNs, e.g., X_train.fillna(X_train.median(), inplace=True)
        
    start_smote_time = time.time()
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    end_smote_time = time.time()

    print(f"SMOTE completed in {end_smote_time - start_smote_time:.2f} seconds.")
    print(f"Original training set shape: {X_train.shape}")
    print(f"Resampled training set shape: {X_train_resampled.shape}")
    print(f"Original class distribution - Normal: {sum(y_train == 0)}, Fraud: {sum(y_train == 1)}")
    print(f"Resampled class distribution - Normal: {sum(y_train_resampled == 0)}, Fraud: {sum(y_train_resampled == 1)}")
else:
    print("Skipping SMOTE as training data is not available.")
    X_train_resampled, y_train_resampled = None, None # Ensure variables exist

In [ ]:
#%% [markdown]
# ## 6. Feature Selection (using Random Forest Importance)
# Use a RandomForest model trained on the resampled data to estimate feature importance.

#%%
if X_train_resampled is not None:
    print("\nPerforming feature selection based on Random Forest importance...")
    feature_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1) # Use multiple cores
    
    start_fs_time = time.time()
    feature_selector.fit(X_train_resampled, y_train_resampled)
    end_fs_time = time.time()
    print(f"Feature importance calculation completed in {end_fs_time - start_fs_time:.2f} seconds.")

    feature_importances = pd.DataFrame({
        'Feature': X_train_resampled.columns,
        'Importance': feature_selector.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    print("\nTop 15 features by importance:")
    print(feature_importances.head(15))

    # Plot feature importances
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=feature_importances.head(15), palette='viridis')
    plt.title('Top 15 Feature Importances (Random Forest)')
    plt.xlabel('Importance Score')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.savefig('feature_importances.png')
    plt.show()

    # Optional: Select top N features (though often XGBoost handles feature selection internally well)
    # N_FEATURES = 20
    # selected_features = feature_importances['Feature'].head(N_FEATURES).tolist()
    # X_train_selected = X_train_resampled[selected_features]
    # X_test_selected = X_test[selected_features]
    # print(f"\nSelected top {N_FEATURES} features.")
    # Use X_train_selected and X_test_selected in subsequent steps if feature selection is applied here.
    # For this example, we will proceed with all features for XGBoost.

else:
    print("Skipping Feature Selection as resampled training data is not available.")

In [ ]:
#%% [markdown]
# ## 7. Model Selection and Training with Modified Rain Optimization

#%%
if X_train_resampled is not None:
    print("\nStep 4: Model Selection and Training with Modified Rain Optimization...")

    # Define hyperparameter grid for XGBoost - Adjust ranges based on experience/domain knowledge
    param_grid = {
        'n_estimators': [100, 200, 300], # More estimators might be needed
        'max_depth': [4, 6, 8, 10],      # Deeper trees might capture complex interactions
        'learning_rate': [0.01, 0.05, 0.1, 0.2], # Controls step size shrinkage
        'subsample': [0.7, 0.8, 0.9],      # Fraction of samples used per tree
        'colsample_bytree': [0.7, 0.8, 0.9], # Fraction of features used per tree
        'min_child_weight': [1, 3, 5],      # Minimum sum of instance weight needed in a child
        'gamma': [0, 0.1, 0.2, 0.3],        # Minimum loss reduction required to make a split
        # 'scale_pos_weight': [1] # Usually not needed after SMOTE, but can be tuned if imbalance persists or SMOTE isn't used
    }

    # Apply Modified Rain Optimization
    print("\nStarting Modified Rain Optimization for XGBoost hyperparameters...")
    start_time = time.time()
    # Reduced iterations/population for faster demonstration; increase for thorough search
    ro = ModifiedRainOptimization(param_grid=param_grid, scoring='roc_auc', n_iterations=15, population_size=8)
    ro.fit(X_train_resampled, y_train_resampled)
    end_time = time.time()

    print(f"\nModified Rain Optimization completed in {end_time - start_time:.2f} seconds")
    print(f"Best parameters found: {ro.best_params_}")
    print(f"Best cross-validation ROC AUC score: {ro.best_score_:.4f}")

    # Visualize the convergence of the optimization algorithm
    plt.figure(figsize=(10, 6))
    history_df = pd.DataFrame(ro.history)
    plt.plot(history_df['iteration'], history_df['best_score'], marker='o', linestyle='-', color='purple')
    plt.title('Modified Rain Optimization Convergence for XGBoost')
    plt.xlabel('Iteration')
    plt.ylabel('Best Score (ROC AUC)')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('rain_optimization_convergence.png')
    plt.show()

else:
    print("Skipping Hyperparameter Optimization as training data is not available.")
    ro = None # Ensure 'ro' exists

In [ ]:
#%% [markdown]
# ## 8. Train Final Model with Optimized Hyperparameters

#%%
if ro is not None and ro.best_params_ is not None and X_train_resampled is not None:
    print("\nTraining the final XGBoost model with optimized hyperparameters...")
    final_model = XGBClassifier(**ro.best_params_, random_state=42, use_label_encoder=False, eval_metric='logloss')

    start_train_time = time.time()
    final_model.fit(X_train_resampled, y_train_resampled)
    end_train_time = time.time()
    print(f"Final model training completed in {end_train_time - start_train_time:.2f} seconds.")

    # Make predictions on the original (unseen) test set
    print("Making predictions on the test set...")
    y_pred_prob = final_model.predict_proba(X_test)[:, 1]
    threshold = 0.5 # Default threshold for initial evaluation
    y_pred = (y_pred_prob >= threshold).astype(int)

else:
    print("Skipping final model training as optimized parameters or data are not available.")
    final_model = None
    y_pred_prob = None
    y_pred = None

In [ ]:
#%% [markdown]
# ## 9. Model Evaluation

#%% [markdown]
# ### 9.1 Classification Metrics and Confusion Matrix (Default Threshold 0.5)

#%%
if final_model is not None and y_pred is not None:
    print("\nStep 5: Model Evaluation...")

    # Classification metrics
    print("\nClassification Report (Threshold = 0.5):")
    print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Fraud (1)']))

    # Confusion Matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Predicted Normal', 'Predicted Fraud'],
                yticklabels=['Actual Normal', 'Actual Fraud'])
    plt.title('Confusion Matrix (Threshold = 0.5)')
    plt.ylabel('Actual Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.savefig('confusion_matrix_default_thresh.png')
    plt.show()
else:
    print("Skipping evaluation as model or predictions are not available.")

In [ ]:
#%% [markdown]
# ### 9.2 ROC Curve and AUC

#%%
if final_model is not None and y_pred_prob is not None:
    fpr, tpr, thresholds_roc = roc_curve(y_test, y_pred_prob)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(10, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'XGBoost ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance level (AUC = 0.50)')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR)')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('roc_curve.png')
    plt.show()

    print(f"\nROC AUC Score: {roc_auc:.4f}")
else:
    print("Skipping ROC Curve plotting as model or predictions are not available.")

In [ ]:
#%% [markdown]
# ### 9.3 Precision-Recall Curve and Average Precision

#%%
if final_model is not None and y_pred_prob is not None:
    precision, recall, thresholds_pr = precision_recall_curve(y_test, y_pred_prob)
    pr_auc = average_precision_score(y_test, y_pred_prob) # Equivalent to AUC PR

    plt.figure(figsize=(10, 6))
    plt.plot(recall, precision, color='blue', lw=2, label=f'XGBoost PR curve (AP = {pr_auc:.4f})')
    # Plot baseline (no-skill classifier)
    no_skill = len(y_test[y_test==1]) / len(y_test)
    plt.plot([0, 1], [no_skill, no_skill], linestyle='--', color='grey', label=f'No Skill (AP = {no_skill:.4f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve')
    plt.legend(loc="best") # Often 'best' or 'upper right' for PR curves
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('precision_recall_curve.png')
    plt.show()

    print(f"Precision-Recall AUC (Average Precision): {pr_auc:.4f}")
else:
    print("Skipping Precision-Recall Curve plotting as model or predictions are not available.")

In [ ]:
#%% [markdown]
# ### 9.4 Finding Optimal Threshold
# Select threshold based on maximizing F1-score for the positive (Fraud) class. Other strategies (e.g., maximizing recall under a precision constraint) could also be used depending on business needs.

#%%
if final_model is not None and y_pred_prob is not None:
    print("\nFinding optimal threshold based on F1-score for Fraud class...")
    f1_scores = []
    thresholds = np.linspace(0.01, 0.99, 100) # Test a range of thresholds

    for thresh in thresholds:
        y_pred_threshold = (y_pred_prob >= thresh).astype(int)
        # Calculate F1 for the positive class (label '1')
        report = classification_report(y_test, y_pred_threshold, output_dict=True, zero_division=0)
        f1_scores.append(report['1']['f1-score']) # F1 score for Fraud class

    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    y_pred_optimal = (y_pred_prob >= optimal_threshold).astype(int)

    print(f"\nOptimal threshold based on F1-score: {optimal_threshold:.4f}")
    print(f"Highest F1-score achieved: {f1_scores[optimal_idx]:.4f}")

    print("\nClassification Report with Optimal Threshold:")
    print(classification_report(y_test, y_pred_optimal, target_names=['Normal (0)', 'Fraud (1)']))

    # Plot threshold vs F1 score
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, f1_scores, marker='.', linestyle='-', color='green')
    plt.scatter(optimal_threshold, f1_scores[optimal_idx], marker='o', color='red', s=100, label=f'Optimal Threshold = {optimal_threshold:.4f}\nMax F1 = {f1_scores[optimal_idx]:.4f}')
    plt.xlabel('Threshold')
    plt.ylabel('F1 Score (Fraud Class)')
    plt.title('Threshold vs F1 Score for Fraud Detection')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('threshold_f1_curve.png')
    plt.show()

    # Evaluate with optimal threshold - Confusion Matrix
    conf_matrix_optimal = confusion_matrix(y_test, y_pred_optimal)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix_optimal, annot=True, fmt='d', cmap='Greens',
                xticklabels=['Predicted Normal', 'Predicted Fraud'],
                yticklabels=['Actual Normal', 'Actual Fraud'])
    plt.title(f'Confusion Matrix (Optimal Threshold = {optimal_threshold:.4f})')
    plt.ylabel('Actual Class')
    plt.xlabel('Predicted Class')
    plt.tight_layout()
    plt.savefig('confusion_matrix_optimal_thresh.png')
    plt.show()

else:
    print("Skipping Optimal Threshold calculation as model or predictions are not available.")
    optimal_threshold = 0.5 # Fallback to default if calculation failed

In [ ]:
#%% [markdown]
# ## 10. Explainable AI - SHAP Values
# SHAP (SHapley Additive exPlanations) helps understand the contribution of each feature to individual predictions and overall model behavior.

#%%
if final_model is not None and 'X_test' in locals():
    print("\nStep 6: Generating Explanations with SHAP...")

    # Create a smaller sample for SHAP explanations (faster computation)
    # Sample from X_test to explain predictions on unseen data
    n_shap_samples = min(500, len(X_test)) # Adjust sample size as needed
    X_sample = X_test.sample(n_shap_samples, random_state=42)
    # y_sample = y_test.loc[X_sample.index] # Target not needed for TreeExplainer

    print(f"Calculating SHAP values for {n_shap_samples} test samples...")
    start_shap_time = time.time()
    # Use TreeExplainer for tree-based models like XGBoost
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_sample) # For classification, this gives SHAP values for the positive class
    end_shap_time = time.time()
    print(f"SHAP value calculation completed in {end_shap_time - start_shap_time:.2f} seconds.")

    # --- SHAP Summary Plot (Bar format - Global Importance) ---
    plt.figure() # Create a new figure context for SHAP plot
    shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
    # Need to get current figure and adjust layout/title *after* SHAP creates the plot
    fig = plt.gcf()
    fig.set_size_inches(10, 8)
    plt.title("SHAP Global Feature Importance (Mean Absolute SHAP Value)")
    plt.tight_layout()
    plt.savefig('shap_feature_importance_bar.png')
    plt.show()


    # --- SHAP Summary Plot (Dot format - Detailed Impact) ---
    plt.figure() # Create a new figure context
    shap.summary_plot(shap_values, X_sample, show=False)
    fig = plt.gcf()
    fig.set_size_inches(10, 8)
    plt.title("SHAP Feature Impact Summary")
    plt.tight_layout()
    plt.savefig('shap_feature_impact_dot.png')
    plt.show()


    # --- SHAP Importance DataFrame ---
    shap_importance = pd.DataFrame({
        'Feature': X_sample.columns,
        'Mean_Abs_SHAP': np.abs(shap_values).mean(axis=0)
    }).sort_values(by='Mean_Abs_SHAP', ascending=False)

    print("\nTop 10 features by Mean Absolute SHAP importance:")
    print(shap_importance.head(10))


    # --- SHAP Dependence Plots ---
    print("\nGenerating SHAP dependence plots for top features...")
    top_features_shap = shap_importance['Feature'].head(3).values
    for feature in top_features_shap:
        plt.figure() # Create a new figure context
        shap.dependence_plot(feature, shap_values, X_sample, interaction_index=None, show=False) # interaction_index='auto' can be slow
        fig = plt.gcf()
        fig.set_size_inches(10, 6)
        plt.title(f"SHAP Dependence Plot for {feature}")
        plt.tight_layout()
        plt.savefig(f'shap_dependence_{feature}.png')
        plt.show()

else:
    print("Skipping SHAP analysis as the final model or test data is not available.")
    explainer = None # Ensure explainer is defined

In [ ]:
#%% [markdown]
# ## 11. Explainable AI - LIME
# LIME (Local Interpretable Model-agnostic Explanations) explains individual predictions by approximating the model locally with a simpler, interpretable model.

#%%
if final_model is not None and X_train_resampled is not None and 'X_test' in locals():
    print("\nStep 7: Generating Explanations with LIME...")

    # Create a LIME explainer
    # It needs the training data (or a sample) to understand feature distributions
    feature_names = list(X_train_resampled.columns)
    
    # Use a smaller sample of training data for LIME explainer if dataset is large
    n_lime_background = min(1000, len(X_train_resampled))
    X_train_lime_sample = X_train_resampled.sample(n_lime_background, random_state=42)

    print(f"Creating LIME explainer using {n_lime_background} background samples...")
    start_lime_setup_time = time.time()
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train_lime_sample.values, # Use numpy array
        feature_names=feature_names,
        class_names=['Normal', 'Fraud'],
        mode='classification',
        # discretize_continuous=True, # Can help with interpretation but might lose nuance
        random_state=42
    )
    end_lime_setup_time = time.time()
    print(f"LIME explainer created in {end_lime_setup_time - start_lime_setup_time:.2f} seconds.")

    # Generate LIME explanations for a few test samples (e.g., some correctly/incorrectly classified fraud cases)
    lime_explanations = []
    num_lime_samples_to_explain = 5

    # Find some fraud cases in the test set for explanation
    fraud_indices_test = y_test[y_test == 1].index
    if len(fraud_indices_test) > 0:
        num_fraud_samples_to_explain = min(num_lime_samples_to_explain, len(fraud_indices_test))
        fraud_sample_indices = np.random.choice(fraud_indices_test, num_fraud_samples_to_explain, replace=False)

        print(f"\nGenerating LIME explanations for {num_fraud_samples_to_explain} fraud instances from the test set...")
        start_lime_exp_time = time.time()
        for i, idx in enumerate(fraud_sample_indices):
            print(f"Explaining instance index: {idx} (Fraud Case {i+1}/{num_fraud_samples_to_explain})")
            instance = X_test.loc[idx].values # Get the specific instance as numpy array
            try:
                 lime_exp = lime_explainer.explain_instance(
                     data_row=instance,
                     predict_fn=final_model.predict_proba, # LIME needs the probability prediction function
                     num_features=10, # Show top 10 features contributing to the prediction
                     top_labels=1 # Explain the top predicted class (usually Fraud if probability > 0.5)
                 )
                 lime_explanations.append(lime_exp)

                 # Plot the explanation
                 fig = lime_exp.as_pyplot_figure()
                 fig.set_size_inches(10, 6)
                 plt.title(f"LIME Explanation for Instance {idx} (Actual: Fraud)")
                 plt.tight_layout()
                 plt.savefig(f'lime_explanation_fraud_{idx}.png')
                 plt.show()
            except Exception as e:
                 print(f"Error generating LIME explanation for instance {idx}: {e}")
        end_lime_exp_time = time.time()
        print(f"LIME explanations generated in {end_lime_exp_time - start_lime_exp_time:.2f} seconds.")

    else:
        print("No fraud instances found in the test set to explain with LIME.")

else:
    print("Skipping LIME analysis as the final model or necessary data is not available.")
    lime_explanations = [] # Ensure variable exists

In [ ]:
#%% [markdown]
# ## 12. Comparing SHAP and LIME Feature Importance
# Aggregate LIME importances across explained instances and compare with global SHAP importance. Note that LIME is local, while SHAP can provide both local and global views. The comparison here uses aggregated local LIME vs global SHAP.

#%%
if explainer is not None and lime_explanations: # Check if both SHAP and LIME ran
    print("\nStep 8: Comparing SHAP and LIME Feature Importance...")

    # Get aggregated LIME importance (average absolute weight across explanations)
    lime_feature_weights = {}
    for exp in lime_explanations:
        # exp.as_list() returns tuples like ('V14 <= -4.61', -0.15)
        for feature_condition, weight in exp.as_list():
            # Extract base feature name (handle conditions like <=, >, etc.)
            feature_name = feature_condition.split(" ")[0]
            if feature_name in feature_names: # Ensure it's a valid feature
                 lime_feature_weights[feature_name] = lime_feature_weights.get(feature_name, 0) + abs(weight)

    if lime_feature_weights:
         # Average the weights over the number of explanations
         lime_importance_df = pd.DataFrame({
             'Feature': list(lime_feature_weights.keys()),
             'Avg_Abs_LIME_Weight': [w / len(lime_explanations) for w in lime_feature_weights.values()]
         }).sort_values(by='Avg_Abs_LIME_Weight', ascending=False)

         print("\nTop 10 Features by Aggregated LIME Importance:")
         print(lime_importance_df.head(10))

         # Merge SHAP (already calculated) and LIME importance
         # Use the SHAP importance calculated earlier (shap_importance DataFrame)
         comparison_df = shap_importance.merge(
             lime_importance_df,
             on='Feature',
             how='outer' # Keep features important in either method
         ).fillna(0)

         # Select top N features based on either metric for plotting
         comparison_df['Max_Importance'] = comparison_df[['Mean_Abs_SHAP', 'Avg_Abs_LIME_Weight']].max(axis=1)
         comparison_df_top = comparison_df.sort_values(by='Max_Importance', ascending=False).head(15)

         # Normalize for better comparison on the plot (0 to 1 scale)
         comparison_df_top['SHAP_Normalized'] = comparison_df_top['Mean_Abs_SHAP'] / comparison_df_top['Mean_Abs_SHAP'].max()
         comparison_df_top['LIME_Normalized'] = comparison_df_top['Avg_Abs_LIME_Weight'] / comparison_df_top['Avg_Abs_LIME_Weight'].max()


         print("\nComparison of SHAP vs Aggregated LIME Feature Importance (Top 15):")
         print(comparison_df_top[['Feature', 'SHAP_Normalized', 'LIME_Normalized']])

         # Visualize the comparison
         plt.figure(figsize=(12, 8))
         comparison_df_melted = pd.melt(comparison_df_top[['Feature', 'SHAP_Normalized', 'LIME_Normalized']],
                                        id_vars=['Feature'],
                                        var_name='Method', value_name='Normalized Importance')
         sns.barplot(y='Feature', x='Normalized Importance', hue='Method', data=comparison_df_melted, palette='magma')
         plt.title('Normalized SHAP vs LIME Feature Importance Comparison')
         plt.xlabel('Normalized Importance Score')
         plt.ylabel('Feature')
         plt.legend(title='Method')
         plt.tight_layout()
         plt.savefig('shap_vs_lime_importance_comparison.png')
         plt.show()
    else:
         print("Could not aggregate LIME importance scores.")

else:
    print("Skipping SHAP vs LIME comparison as one or both analyses were not performed.")

In [ ]:
#%% [markdown]
# ## 13. Fraud Detection System Implementation
# Define a function that takes a new transaction, preprocesses it, predicts fraud using the trained model and optimal threshold, and provides a SHAP-based explanation.

#%%
if final_model is not None and explainer is not None and 'scaler' in locals() and 'X_train_resampled' in locals() and optimal_threshold is not None:
    print("\nStep 9: Implementing a Fraud Detection System Function...")

    # Get the required feature order from the training data
    MODEL_FEATURES = X_train_resampled.columns.tolist()

    def fraud_detection_system(transaction_df, model, threshold, shap_explainer, scaler_obj, feature_order):
        """
        Detects if a transaction is fraudulent and explains the decision using SHAP.

        Parameters:
        transaction_df (pd.DataFrame): A DataFrame with a single row representing the transaction.
                                       Must contain 'Time' and 'Amount' columns initially.
        model: Trained XGBoost model.
        threshold (float): Classification threshold for fraud.
        shap_explainer: Trained SHAP TreeExplainer.
        scaler_obj: Fitted StandardScaler for the 'Amount' feature.
        feature_order (list): List of feature names in the order the model expects.

        Returns:
        dict: Detection result including probability, flag, confidence, and SHAP explanation.
        """
        if not isinstance(transaction_df, pd.DataFrame) or transaction_df.shape[0] != 1:
            raise ValueError("Input 'transaction_df' must be a pandas DataFrame with exactly one row.")
        if 'Time' not in transaction_df.columns or 'Amount' not in transaction_df.columns:
             raise ValueError("Input DataFrame must contain 'Time' and 'Amount' columns.")

        # Preprocess the transaction (make a copy to avoid modifying original)
        transaction_processed = transaction_df.copy()

        # 1. Scale Amount
        transaction_processed['Amount_Scaled'] = scaler_obj.transform(transaction_processed[['Amount']])

        # 2. Create Time features
        seconds_in_day = 24 * 60 * 60
        transaction_processed['Time_sin'] = np.sin(2 * np.pi * transaction_processed['Time'] / seconds_in_day)
        transaction_processed['Time_cos'] = np.cos(2 * np.pi * transaction_processed['Time'] / seconds_in_day)

        # 3. Drop original Time and Amount
        transaction_processed.drop(['Time', 'Amount'], axis=1, inplace=True)

        # 4. Ensure all model features are present and in the correct order
        # Add missing columns with 0 (assuming missing features weren't present)
        missing_cols = set(feature_order) - set(transaction_processed.columns)
        for col in missing_cols:
            transaction_processed[col] = 0
        # Reorder columns to match model's training order
        transaction_processed = transaction_processed[feature_order]

        # Make prediction
        fraud_prob = model.predict_proba(transaction_processed)[0, 1]
        is_fraud = fraud_prob >= threshold

        # Generate explanation using SHAP
        shap_values_instance = shap_explainer.shap_values(transaction_processed) # SHAP values for the positive class

        # Get the top contributing features for this specific instance
        feature_importance = pd.DataFrame({
            'Feature': feature_order,
            'Feature_Value': transaction_processed.values[0],
            'SHAP_Value': shap_values_instance[0] # Impact on the positive class prediction
        })
        feature_importance['Abs_SHAP_Value'] = feature_importance['SHAP_Value'].abs()
        feature_importance = feature_importance.sort_values(by='Abs_SHAP_Value', ascending=False)

        # Format the result
        result = {
            'transaction_id': transaction_df.index[0] if transaction_df.index is not None else 'N/A',
            'fraud_probability': float(fraud_prob),
            'is_fraud': bool(is_fraud),
            'threshold_used': float(threshold),
            'confidence': float(abs(fraud_prob - 0.5) * 2), # Simple confidence measure (0-1)
            'explanation': {
                'shap_base_value': float(shap_explainer.expected_value), # Average prediction probability
                'top_features': feature_importance.head(5).to_dict(orient='records'),
            }
        }

        return result

    print("Fraud detection system function defined.")

    # --- Example Usage ---
    # Take the first transaction from the original *unprocessed* test split
    # Need to get the original 'Time' and 'Amount' back for the function input
    if data is not None and not X_test.empty:
         test_index_0 = X_test.index[0]
         # Create a DataFrame for the single transaction with original Time/Amount
         sample_transaction_original = data.loc[[test_index_0]].drop('Class', axis=1)

         print("\nApplying fraud detection system to a sample transaction (Index: {}):".format(test_index_0))
         print("Original Sample Transaction Data:")
         print(sample_transaction_original)

         try:
             detection_result = fraud_detection_system(
                 transaction_df=sample_transaction_original,
                 model=final_model,
                 threshold=optimal_threshold,
                 shap_explainer=explainer,
                 scaler_obj=scaler,
                 feature_order=MODEL_FEATURES
             )

             print("\n--- Fraud Detection Result ---")
             print(f"Transaction ID: {detection_result['transaction_id']}")
             print(f"Fraud Probability: {detection_result['fraud_probability']:.4f}")
             print(f"Is Fraudulent (Threshold {detection_result['threshold_used']:.4f}): {detection_result['is_fraud']}")
             print(f"Confidence Score: {detection_result['confidence']:.4f}")
             print("\nExplanation (Top 5 contributing features based on SHAP):")
             print(f"  - Base Probability (Average): {detection_result['explanation']['shap_base_value']:.4f}")
             for feature_info in detection_result['explanation']['top_features']:
                  impact_direction = "increases" if feature_info['SHAP_Value'] > 0 else "decreases"
                  print(f"  - Feature: {feature_info['Feature']:<15} | Value: {feature_info['Feature_Value']:<10.4f} | SHAP Value: {feature_info['SHAP_Value']:<10.4f} ({impact_direction} fraud score)")
         except Exception as e:
              print(f"\nError running fraud detection system on sample: {e}")

    else:
         print("\nSkipping example usage as test data or original data is not available.")

else:
    print("Skipping Fraud Detection System implementation as necessary components are missing.")

In [ ]:
#%% [markdown]
# ## 14. Save Model and Components
# Save the trained model, scaler, SHAP explainer, optimal threshold, and feature list for future use in deployment.

#%%
print("\nStep 10: Save the trained model and necessary components...")

if final_model is not None and 'scaler' in locals() and explainer is not None and optimal_threshold is not None and 'MODEL_FEATURES' in locals():
    try:
        joblib.dump(final_model, 'fraud_detection_model.pkl')
        joblib.dump(scaler, 'amount_scaler.pkl')
        joblib.dump(explainer, 'shap_explainer.pkl')
        joblib.dump(optimal_threshold, 'optimal_threshold.pkl')
        joblib.dump(MODEL_FEATURES, 'model_features.pkl')

        print("\nComponents saved successfully:")
        print("- fraud_detection_model.pkl")
        print("- amount_scaler.pkl")
        print("- shap_explainer.pkl")
        print("- optimal_threshold.pkl")
        print("- model_features.pkl")
    except Exception as e:
        print(f"\nError saving components: {e}")
else:
    print("\nSkipping saving components as some are missing.")

print("\n--- Credit Card Fraud Detection Notebook Completed ---")

#%%